In [ ]:
import numpy as np
import evaluate
import cv2
from tqdm import tqdm

from segment_enhanced_model import SegmentEnhancedModel
from src.utils.dataset import load_foodseg103_splits
from src.constants.category_id import CATEGORY_ID

In [2]:
image_dataset = load_foodseg103_splits()["validation"]

In [ ]:
enhanced_model = SegmentEnhancedModel(sam_model_type="vit_h")
val_mean_iou = evaluate.load("mean_iou")

In [4]:
for idx in tqdm(range(2)):
    image_information = image_dataset[idx]
    image = np.array(image_information["image"])
    label = np.array(image_information["label"])
    
    original_size = label.shape[:2]
    image = cv2.resize(image, (512, 512))
    mask = enhanced_model.predict_segmentation_mask(image)
    mask = mask.astype(np.uint8)
    mask = cv2.resize(mask, (original_size[1], original_size[0]), interpolation=cv2.INTER_NEAREST)
    val_mean_iou.add_batch(predictions=mask, references=label)

100%|██████████| 2/2 [01:52<00:00, 56.24s/it]


In [5]:
results = val_mean_iou.compute(
    ignore_index=0,
    reduce_labels=False,
    num_labels=len(CATEGORY_ID)
)

In [6]:
results

{'mean_iou': 0.48962836794824643,
 'mean_accuracy': 0.715484999483376,
 'overall_accuracy': 0.7262722191060135,
 'per_category_iou': array([0.        ,        nan,        nan,        nan,        nan,
               nan,        nan,        nan, 0.93794317,        nan,
        0.55460695,        nan,        nan,        nan,        nan,
               nan,        nan,        nan,        nan,        nan,
               nan,        nan,        nan,        nan,        nan,
               nan,        nan,        nan,        nan, 0.95559172,
               nan,        nan,        nan,        nan,        nan,
               nan,        nan,        nan,        nan,        nan,
               nan,        nan,        nan,        nan,        nan,
               nan,        nan,        nan,        nan,        nan,
               nan,        nan,        nan,        nan,        nan,
               nan,        nan,        nan, 0.        ,        nan,
               nan,        nan,        nan,        n

### References
---

[1] https://colab.research.google.com/github/NielsRogge/Transformers-Tutorials/blob/master/MaskFormer/Fine-tuning/Fine_tuning_MaskFormerForInstanceSegmentation_on_semantic_sidewalk.ipynb#scrollTo=tXCWTfTbz5Wu